In [8]:
"""
Data Preparation — Vehicle Insurance Fraud Dataset
Cleaning + Encoding, with a class-imbalance check
(full EDA skipped per assignment scope)

Update DATA_PATH below to point to your CSV.
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

pd.set_option('display.max_columns', None)

# ----------------------------------------------------------------------
# 1. LOAD DATA
# ----------------------------------------------------------------------
DATA_PATH = "carclaims.csv"  # <-- change to your file path
df = pd.read_csv(DATA_PATH)
print("Loaded shape:", df.shape)

# ----------------------------------------------------------------------
# 2. CLEAN TARGET: FraudFound -> 0/1
# ----------------------------------------------------------------------
print("\nRaw FraudFound values:", df['FraudFound'].unique())

cleaned = df['FraudFound'].astype(str).str.strip().str.lower()
target_map = {'yes': 1, 'no': 0, 'y': 1, 'n': 0, 'true': 1, 'false': 0, '1': 1, '0': 0}
mapped = cleaned.map(target_map)

unmatched = cleaned[mapped.isnull()].unique()
if len(unmatched) > 0:
    raise ValueError(f"Unmapped FraudFound values: {unmatched}. Extend target_map above.")

df['FraudFound'] = mapped.astype(int)
print("Cleaned FraudFound values:", df['FraudFound'].unique())

# ----------------------------------------------------------------------
# 3. CLASS IMBALANCE CHECK (needed to decide loss weighting strategy)
# ----------------------------------------------------------------------
print("\n" + "=" * 60)
print("CLASS BALANCE CHECK")
print("=" * 60)
class_counts = df['FraudFound'].value_counts()
class_pct = df['FraudFound'].value_counts(normalize=True) * 100
print(pd.DataFrame({'count': class_counts, 'pct': class_pct.round(2)}))

fraud_rate = df['FraudFound'].mean()
imbalance_ratio = (1 - fraud_rate) / fraud_rate
print(f"\nFraud rate: {fraud_rate*100:.2f}%")
print(f"Imbalance ratio (non-fraud : fraud) = {imbalance_ratio:.1f} : 1")

if imbalance_ratio > 3:
    print(">>> Significant imbalance detected. Plan: use class-weighted loss "
          "(or focal loss) during training, and evaluate with PR-AUC / F1 / "
          "recall rather than accuracy.")

# Class weights to use later during model.fit()
n_samples = len(df)
n_classes = 2
class_weight = {
    0: n_samples / (n_classes * class_counts[0]),
    1: n_samples / (n_classes * class_counts[1]),
}
print("\nSuggested class_weight dict for training:", class_weight)

# ----------------------------------------------------------------------
# 4. DROP NON-PREDICTIVE / IDENTIFIER COLUMNS
# ----------------------------------------------------------------------
drop_cols = ['PolicyNumber']  # pure identifier, no predictive signal
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
print(f"\nDropped identifier columns: {drop_cols}")

# Check for exact duplicate rows
dup_count = df.duplicated().sum()
print(f"Duplicate rows found: {dup_count}")
if dup_count > 0:
    df = df.drop_duplicates()
    print(f"Dropped duplicates. New shape: {df.shape}")

# ----------------------------------------------------------------------
# 5. HANDLE PLACEHOLDER / SENTINEL VALUES
# ----------------------------------------------------------------------
# This dataset uses '0' as a sentinel for "no prior policyholder age data"
# in the Age column for some rows. Check and flag rather than silently keep.
print("\n" + "=" * 60)
print("PLACEHOLDER VALUE CHECK")
print("=" * 60)
if 'Age' in df.columns:
    zero_age_count = (df['Age'] == 0).sum()
    print(f"Rows with Age == 0: {zero_age_count} ({zero_age_count/len(df)*100:.2f}%)")
    if zero_age_count > 0:
        # Impute with median age of the matching AgeOfPolicyHolder bucket if available,
        # else overall median — simplest defensible approach given time constraints
        median_age = df.loc[df['Age'] != 0, 'Age'].median()
        df.loc[df['Age'] == 0, 'Age'] = median_age
        print(f"Imputed Age==0 with overall median age: {median_age}")

# ----------------------------------------------------------------------
# 6. ENCODE ORDINAL (BUCKETED) COLUMNS AS INTEGERS
# ----------------------------------------------------------------------
# These columns are text but represent a genuine order — encoding them
# as ordinal integers preserves that structure (better than plain one-hot
# or an unordered embedding, and cheap to justify in a report).

ordinal_maps = {
    'AgeOfVehicle': {
        'new': 0, '2 years': 1, '3 years': 2, '4 years': 3, '5 years': 4,
        '6 years': 5, '7 years': 6, 'more than 7': 7
    },
    'AgeOfPolicyHolder': {
        '16 to 17': 0, '18 to 20': 1, '21 to 25': 2, '26 to 30': 3,
        '31 to 35': 4, '36 to 40': 5, '41 to 50': 6, '51 to 65': 7,
        'over 65': 8
    },
    'PastNumberOfClaims': {
        'none': 0, '1': 1, '2 to 4': 2, 'more than 4': 3
    },
    'NumberOfSuppliments': {
        'none': 0, '1 to 2': 1, '3 to 5': 2, 'more than 5': 3
    },
    'NumberOfCars': {
        '1 vehicle': 0, '2 vehicles': 1, '3 to 4': 2, '5 to 8': 3, 'more than 8': 4
    },
    'AddressChange-Claim': {
        'no change': 0, 'under 6 months': 1, '1 year': 2, '2 to 3 years': 3,
        '4 to 8 years': 4
    },
    'Days:Policy-Accident': {
        'none': 0, '1 to 7': 1, '8 to 15': 2, '15 to 30': 3, 'more than 30': 4
    },
    'Days:Policy-Claim': {
        'none': 0, '8 to 15': 1, '15 to 30': 2, 'more than 30': 3
    },
}

print("\n" + "=" * 60)
print("ORDINAL ENCODING")
print("=" * 60)
for col, mapping in ordinal_maps.items():
    if col not in df.columns:
        continue
    original_vals = df[col].astype(str).str.strip().unique()
    unmapped = [v for v in original_vals if v not in mapping]
    if unmapped:
        print(f"WARNING: '{col}' has values not in mapping: {unmapped} "
              f"— add them to ordinal_maps before proceeding.")
        continue
    df[col] = df[col].astype(str).str.strip().map(mapping)
    print(f"Encoded '{col}' -> integer scale 0-{max(mapping.values())}")

# ----------------------------------------------------------------------
# 7. ENCODE REMAINING NOMINAL CATEGORICAL COLUMNS (label encoding,
#    to be fed through embedding layers in the neural net)
# ----------------------------------------------------------------------
nominal_cols = df.select_dtypes(include=['object']).columns.tolist()
print("\n" + "=" * 60)
print(f"NOMINAL CATEGORICAL COLUMNS TO LABEL-ENCODE ({len(nominal_cols)}):")
print(nominal_cols)
print("=" * 60)

encoders = {}
cardinalities = {}
for col in nominal_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str).str.strip())
    encoders[col] = le
    cardinalities[col] = len(le.classes_)
    print(f"'{col}': {cardinalities[col]} categories")

print("\nCardinalities (needed to size embedding layers later):")
print(cardinalities)

# ----------------------------------------------------------------------
# 8. TRAIN / VAL / TEST SPLIT (stratified — imbalance-safe)
# ----------------------------------------------------------------------
X = df.drop(columns=['FraudFound'])
y = df['FraudFound']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print("\n" + "=" * 60)
print("SPLIT SIZES")
print("=" * 60)
print(f"Train: {X_train.shape}, fraud rate: {y_train.mean()*100:.2f}%")
print(f"Val:   {X_val.shape}, fraud rate: {y_val.mean()*100:.2f}%")
print(f"Test:  {X_test.shape}, fraud rate: {y_test.mean()*100:.2f}%")

# ----------------------------------------------------------------------
# 9. SCALE CONTINUOUS / ORDINAL FEATURES (fit on train only)
# ----------------------------------------------------------------------
# Scale everything except the truly high-cardinality nominal columns that
# will go through embeddings (those should stay as raw integer codes).
embedding_cols = [c for c in nominal_cols if cardinalities[c] > 2]
scale_cols = [c for c in X.columns if c not in embedding_cols]

scaler = StandardScaler()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_val[scale_cols] = scaler.transform(X_val[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

print("\n" + "=" * 60)
print("FEATURE ROUTING FOR THE NEURAL NET")
print("=" * 60)
print(f"Columns to embed (categorical, cardinality > 2): {embedding_cols}")
print(f"Columns to feed as scaled numeric input: {scale_cols}")

# ----------------------------------------------------------------------
# 10. SAVE PROCESSED SPLITS
# ----------------------------------------------------------------------
X_train.to_csv("X_train.csv", index=False)
X_val.to_csv("X_val.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_val.to_csv("y_val.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("\nSaved: X_train.csv, X_val.csv, X_test.csv, y_train.csv, y_val.csv, y_test.csv")
print("Data preparation complete.")

Loaded shape: (15420, 33)

Raw FraudFound values: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Cleaned FraudFound values: [0 1]

CLASS BALANCE CHECK
            count    pct
FraudFound              
0           14497  94.01
1             923   5.99

Fraud rate: 5.99%
Imbalance ratio (non-fraud : fraud) = 15.7 : 1
>>> Significant imbalance detected. Plan: use class-weighted loss (or focal loss) during training, and evaluate with PR-AUC / F1 / recall rather than accuracy.

Suggested class_weight dict for training: {0: np.float64(0.5318341725874319), 1: np.float64(8.353196099674973)}

Dropped identifier columns: ['PolicyNumber']
Duplicate rows found: 0

PLACEHOLDER VALUE CHECK
Rows with Age == 0: 320 (2.08%)
Imputed Age==0 with overall median age: 39.0

ORDINAL ENCODING
Encoded 'AgeOfVehicle' -> integer scale 0-7
Encoded 'AgeOfPolicyHolder' -> integer scale 0-8
Encoded 'PastNumberOfClaims' -> integer scale 0-3
Encoded 'NumberOfSuppliments' -> integer scale 0-3
Encoded 'NumberOfCars' 

C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_16544\884920416.py:157: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  nominal_cols = df.select_dtypes(include=['object']).columns.tolist()



Saved: X_train.csv, X_val.csv, X_test.csv, y_train.csv, y_val.csv, y_test.csv
Data preparation complete.
